# Module 2: Recall Layer - Candidate Generation (MBA + ALS)

## Objective
Construct a high-recall candidate set (100-200 items per user) by combining:
1. **Organic Market Basket Analysis (MBA)** for familiarity signal.
2. **Implicit ALS Collaborative Filtering** for discovery signal.

## Methodological Principles
- Use **organic baskets** for MBA to reduce promotion-induced confounding.
- Use **retailer-revenue-weighted confidence** for ALS: $c_{ui} = 1 + \log_{10}(\text{Revenue\_Retailer}_{ui}+1)$.
- Normalize both recall signals to $[0,1]$ for downstream utility integration.
- Enforce initial uplift guardrail by excluding items purchased in the recent 4 weeks.

## Deliverables
- `association_rules.csv` with normalized MBA relevance.
- `user_item_matrix.npz` for ALS and downstream reproducibility.
- `candidate_set_module2.csv` with unioned, deduplicated, normalized relevance scores.

This notebook is written defensively: if processed parquet artifacts are unavailable (for example due to Git LFS pointers), it reconstructs required tables from raw CSV files.

In [1]:
from pathlib import Path
import importlib
import sys
import warnings

import numpy as np
import pandas as pd
from scipy.sparse import save_npz

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.data_loader as data_loader_module
import src.recall_engine as recall_engine_module

importlib.reload(data_loader_module)
importlib.reload(recall_engine_module)

from src.data_loader import load_or_build_master_transactions
from src.recall_engine import build_als_model, build_candidate_set, build_mba_rules, save_als_factors

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

# ---- Reproducibility ----
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ---- Runtime controls ----
SAMPLE_NROWS = 1_000_000      # Set to None for full corpus run
CANDIDATE_USERS_LIMIT = None   # None for all users
TOP_ALS = 50
TOP_MBA = 50
SEED_ITEMS_K = 3
TOP_ALS_EXPORT = 100          # Task 2: Keep top-100 ALS scores per user
ALS_SCORE_BATCH_SIZE = 256

# ---- Project paths (robust to notebook launch location) ----
DATA_RAW = PROJECT_ROOT / "data" / "01_raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "02_processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_RAW exists:", DATA_RAW.exists())
print("DATA_PROCESSED exists:", DATA_PROCESSED.exists())

PROJECT_ROOT: D:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_full\project_ddm_complete_journey
DATA_RAW exists: True
DATA_PROCESSED exists: True


In [2]:
tx_all, tx_org = load_or_build_master_transactions(
    raw_dir=DATA_RAW,
    processed_dir=DATA_PROCESSED,
    sample_nrows=SAMPLE_NROWS,
)

print("all transactions shape:", tx_all.shape)
print("organic-only transactions shape:", tx_org.shape)
print("households:", tx_all["household_key"].nunique())
print("baskets:", tx_all["BASKET_ID"].nunique())
print("commodities:", tx_all["COMMODITY_DESC"].nunique())
print("day range:", int(tx_all["DAY"].min()), "->", int(tx_all["DAY"].max()))

promo_basket_share = tx_all.groupby("BASKET_ID")["Is_Promoted_Item"].any().mean()
print(f"share of promoted baskets: {promo_basket_share:.3f}")

display(tx_all.head(3))
display(tx_org.head(3))

all transactions shape: (2216946, 31)
organic-only transactions shape: (85487, 31)
households: 2500
baskets: 275889
commodities: 307
day range: 1 -> 711
share of promoted baskets: 0.837


,household_key,BASKET_ID,DAY,WEEK_NO,STORE_ID,TRANS_TIME,COMMODITY_DESC,DEPARTMENT,SUB_COMMODITY_DESC,BRAND,classification_1,classification_2,classification_3,classification_4,classification_5,HOMEOWNER_DESC,KID_CATEGORY_DESC,Total_Quantity,Revenue_Retailer_Total,Price_Paid_Customer_Total,Retail_Discount_Total,Coupon_Discount_Total,Coupon_Match_Discount_Total,Normalized_Margin,Distinct_SKU_Count,Distinct_Subcommodity_Count,Revenue_Retailer_Outlier_Count,Is_Promoted_Item,Basket_Is_Organic,Revenue_Retailer,Price_Paid_Customer
0,1,27601281299,51,8,436,1456,BACON,MEAT-PCKGD,PRE-COOKED,National,Age Group6,X,Level4,2,Group5,Homeowner,None/Unknown,1,2.50,1.51,0.99,0.0,0.0,1.0,1,1,0,True,False,2.50,1.51
1,1,27601281299,51,8,436,1456,BAKED BREAD/BUNS/ROLLS,GROCERY,FRUIT/BREAKFAST BREAD,National,Age Group6,X,Level4,2,Group5,Homeowner,None/Unknown,1,2.50,2.01,0.49,0.0,0.0,1.0,1,1,0,True,False,2.50,2.01
2,1,27601281299,51,8,436,1456,BATH TISSUES,GROCERY,TOILET TISSUE,National,Age Group6,X,Level4,2,Group5,Homeowner,None/Unknown,1,7.19,5.89,1.30,0.0,0.0,1.0,1,1,0,True,False,7.19,5.89


,household_key,BASKET_ID,DAY,WEEK_NO,STORE_ID,TRANS_TIME,COMMODITY_DESC,DEPARTMENT,SUB_COMMODITY_DESC,BRAND,classification_1,classification_2,classification_3,classification_4,classification_5,HOMEOWNER_DESC,KID_CATEGORY_DESC,Total_Quantity,Revenue_Retailer_Total,Price_Paid_Customer_Total,Retail_Discount_Total,Coupon_Discount_Total,Coupon_Match_Discount_Total,Normalized_Margin,Distinct_SKU_Count,Distinct_Subcommodity_Count,Revenue_Retailer_Outlier_Count,Is_Promoted_Item,Basket_Is_Organic,Revenue_Retailer,Price_Paid_Customer
0,1,30578771955,235,34,436,1629,BREAKFAST SWEETS,PASTRY,SW GDS: SW ROLLS/DAN,National,Age Group6,X,Level4,2,Group5,Homeowner,None/Unknown,1,1.19,1.19,0.0,0.0,0.0,1.0,1,1,0,False,True,1.19,1.19
1,1,30700731022,246,36,436,1107,COFFEE SHOP,RESTAURANT,SV BEV: COFFEE DINE IN,National,Age Group6,X,Level4,2,Group5,Homeowner,None/Unknown,1,1.39,1.39,0.0,0.0,0.0,0.0,1,1,0,False,True,1.39,1.39
2,1,32556765955,375,54,436,1225,COFFEE SHOP,RESTAURANT,SV BEV: COFFEE DINE IN,National,Age Group6,X,Level4,2,Group5,Homeowner,None/Unknown,1,1.39,1.39,0.0,0.0,0.0,0.0,1,1,0,False,True,1.39,1.39


In [3]:
mba_rules_long, mba_rules_raw = build_mba_rules(tx_org)
print("Expanded MBA rules:", len(mba_rules_long))
display(mba_rules_long.head(10))

if mba_rules_long.empty:
    print("MBA rules are empty for this sample; downstream candidate generation will rely on ALS only.")

Expanded MBA rules: 526


,antecedent_item,consequent_item,lift,confidence,support,relevance_mba
0,ANALGESICS,COLD AND FLU,5.990339,0.112903,0.001718,1.000000
1,ANALGESICS,FIRST AID PRODUCTS,5.349512,0.067742,0.001031,1.000000
2,ANALGESICS,ORAL HYGIENE PRODUCTS,3.876894,0.067742,0.001031,1.000000
3,APPLES,CITRUS,5.821730,0.112583,0.001669,1.000000
4,APPLES,TROPICAL FRUIT,5.808122,0.258278,0.003828,1.000000
5,APPLES,WATER - CARBONATED/FLVRD DRINK,1.740996,0.079470,0.001178,0.370498
6,APPLES,SALAD BAR,1.629555,0.082781,0.001227,0.314777
7,APPLES,FLUID MILK PRODUCTS,1.102947,0.119205,0.001767,0.051473
8,BABY FOODS,BABY HBC,9.378222,0.100806,0.001227,1.000000
9,BABY FOODS,DIAPERS & DISPOSABLES,8.660732,0.141129,0.001718,1.000000


In [4]:
(
    als_model,
    user_item_matrix,
    user_to_idx,
    idx_to_user,
    item_to_idx,
    idx_to_item,
    users,
    items,
    user_factors,
    item_factors,
) = build_als_model(tx_all)

user_factor_path, item_factor_path = save_als_factors(
    output_dir=DATA_PROCESSED,
    users=users,
    items=items,
    user_factors=user_factors,
    item_factors=item_factors,
    user_to_idx=user_to_idx,
    item_to_idx=item_to_idx,
)

save_npz(DATA_PROCESSED / "user_item_matrix.npz", user_item_matrix)
print("ALS matrix shape (users x items):", user_item_matrix.shape)
print("ALS non-zero interactions:", user_item_matrix.nnz)
print("ALS user factor shape:", user_factors.shape)
print("ALS item factor shape:", item_factors.shape)
print("Saved:")
print(" -", user_factor_path)
print(" -", item_factor_path)
print(" -", DATA_PROCESSED / "user_item_matrix.npz")

  0%|          | 0/20 [00:00<?, ?it/s]

ALS matrix shape (users x items): (2500, 307)
ALS non-zero interactions: 286548
ALS user factor shape: (2500, 64)
ALS item factor shape: (307, 64)
Saved:
 - D:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_full\project_ddm_complete_journey\data\02_processed\user_factors.npz
 - D:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_full\project_ddm_complete_journey\data\02_processed\item_factors.npz
 - D:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_full\project_ddm_complete_journey\data\02_processed\user_item_matrix.npz


In [5]:
(
    als_model,
    user_item_matrix,
    user_to_idx,
    idx_to_user,
    item_to_idx,
    idx_to_item,
    users,
    items,
    user_factors,
    item_factors,
) = build_als_model(tx_all)

user_factor_path, item_factor_path = save_als_factors(
    output_dir=DATA_PROCESSED,
    users=users,
    items=items,
    user_factors=user_factors,
    item_factors=item_factors,
    user_to_idx=user_to_idx,
    item_to_idx=item_to_idx,
)

save_npz(DATA_PROCESSED / "user_item_matrix.npz", user_item_matrix)
print("ALS matrix shape (users x items):", user_item_matrix.shape)
print("ALS non-zero interactions:", user_item_matrix.nnz)
print("ALS user factor shape:", user_factors.shape)
print("ALS item factor shape:", item_factors.shape)
print("Saved:")
print(" -", user_factor_path)
print(" -", item_factor_path)
print(" -", DATA_PROCESSED / "user_item_matrix.npz")

  0%|          | 0/20 [00:00<?, ?it/s]

ALS matrix shape (users x items): (2500, 307)
ALS non-zero interactions: 286548
ALS user factor shape: (2500, 64)
ALS item factor shape: (307, 64)
Saved:
 - D:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_full\project_ddm_complete_journey\data\02_processed\user_factors.npz
 - D:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_full\project_ddm_complete_journey\data\02_processed\item_factors.npz
 - D:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_full\project_ddm_complete_journey\data\02_processed\user_item_matrix.npz


In [6]:
# Build the recall-layer artifacts from the shared src implementation.
artifacts = build_candidate_set(
    tx_all=tx_all,
    mba_rules_long=mba_rules_long,
    users=users,
    user_to_idx=user_to_idx,
    user_factors=user_factors,
    item_factors=item_factors,
    items=items,
    top_als=TOP_ALS,
    top_mba=TOP_MBA,
    seed_items_k=SEED_ITEMS_K,
    candidate_users_limit=CANDIDATE_USERS_LIMIT,
    als_score_batch_size=ALS_SCORE_BATCH_SIZE,
)

candidate_set = artifacts.candidate_set
filtered_items_log = artifacts.filtered_items_log
als_scores = artifacts.als_scores
seed_summary = artifacts.seed_summary
recent_pairs = artifacts.recent_pairs

als_scores_path = DATA_PROCESSED / "als_scores.parquet"
als_scores.to_parquet(als_scores_path, index=False)

mba_rules_long.to_csv(DATA_PROCESSED / "association_rules.csv", index=False)
candidate_set.to_csv(DATA_PROCESSED / "candidate_set_module2.csv", index=False)
filtered_items_log.to_csv(DATA_PROCESSED / "filtered_items_log.csv", index=False)

print("ALS top-100 score rows:", len(als_scores))
print("Saved:", als_scores_path)
print("Saved:", DATA_PROCESSED / "association_rules.csv")
print("Saved:", DATA_PROCESSED / "candidate_set_module2.csv")
print("Saved:", DATA_PROCESSED / "filtered_items_log.csv")
print("users selected:", candidate_set["household_key"].nunique())
print("candidate rows:", len(candidate_set))
print("avg candidates per user:", round(candidate_set.groupby("household_key").size().mean(), 2))
print("source mix:")
print(candidate_set["source_detail"].value_counts(normalize=True).round(3))

display(candidate_set.head(20))
display(filtered_items_log.head(20))

ALS top-100 score rows: 125000
Saved: D:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_full\project_ddm_complete_journey\data\02_processed\als_scores.parquet
Saved: D:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_full\project_ddm_complete_journey\data\02_processed\association_rules.csv
Saved: D:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_full\project_ddm_complete_journey\data\02_processed\candidate_set_module2.csv
Saved: D:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_full\project_ddm_complete_journey\data\02_processed\filtered_items_log.csv
users selected: 2500
candidate rows: 117547
avg candidates per user: 47.02
source mix:
source_detail
ALS        0.681
MBA        0.179
BOTH       0.126
UNKNOWN    0.014
Name: proportion, dtype: float64


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source,source_detail,snapshot_week,was_recently_purchased
2,1,BAKED SWEET GOODS,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.192512,0.084372,0.192512,both,BOTH,102,False
3,1,BAKING MIXES,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.444106,0.000000,0.444106,als,ALS,102,False
4,1,BAKING NEEDS,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.282118,0.153670,0.282118,both,BOTH,102,False
7,1,BEEF,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.000000,0.796527,0.796527,mba,MBA,102,False
8,1,BLEACH,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.721154,0.000000,0.721154,als,ALS,102,False
9,1,BREAKFAST SWEETS,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.147806,0.000000,0.147806,als,ALS,102,False
10,1,CANDY - CHECKLANE,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.000000,0.301475,0.301475,mba,MBA,102,False
14,1,CHRISTMAS SEASONAL,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.523417,0.000000,0.523417,als,ALS,102,False
20,1,DISHWASH DETERGENTS,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.251796,0.000000,0.251796,als,ALS,102,False
21,1,DRY BN/VEG/POTATO/RICE,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.202097,0.000000,0.202097,als,ALS,102,False


,household_key,COMMODITY_DESC,filter_reason,reference_week
0,1,ANALGESICS,recent_purchase,102
1,1,BAKED BREAD/BUNS/ROLLS,recent_purchase,102
5,1,BATH TISSUES,recent_purchase,102
6,1,BEANS - CANNED GLASS & MW,recent_purchase,102
11,1,CANDY - PACKAGED,recent_purchase,102
12,1,CHEESE,recent_purchase,102
13,1,CHEESES,recent_purchase,102
15,1,COFFEE,recent_purchase,102
16,1,COLD CEREAL,recent_purchase,102
17,1,COOKIES/CONES,recent_purchase,102


In [7]:
if candidate_set.empty:
    print("Candidate set is empty after filtering.")
else:
    source_pct = candidate_set["source_detail"].value_counts(normalize=True).mul(100).round(2)
    print("Source composition (%):")
    for key in ["ALS", "MBA", "BOTH"]:
        print(f" - {key}: {source_pct.get(key, 0.0)}%")

    print("\nCandidate count per user quantiles:")
    coverage = candidate_set.groupby("household_key")["COMMODITY_DESC"].nunique()
    print(coverage.quantile([0.0, 0.25, 0.5, 0.75, 1.0]).round(2))

    print("\nSample top-5 recommendations for 3 users:")
    for hh in candidate_set["household_key"].drop_duplicates().head(3):
        print(f"\nHousehold {hh}")
        display(candidate_set[candidate_set["household_key"] == hh].sort_values("Relevance", ascending=False).head(5))

Source composition (%):
 - ALS: 68.06%
 - MBA: 17.87%
 - BOTH: 12.63%

Candidate count per user quantiles:
0.00    10.0
0.25    38.0
0.50    48.0
0.75    55.0
1.00    82.0
Name: COMMODITY_DESC, dtype: float64

Sample top-5 recommendations for 3 users:

Household 1


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source,source_detail,snapshot_week,was_recently_purchased
59,1,VEGETABLES - ALL OTHERS,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.000000,1.000000,1.000000,mba,MBA,102,False
47,1,POPCORN,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.947021,0.000000,0.947021,als,ALS,102,False
7,1,BEEF,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.000000,0.796527,0.796527,mba,MBA,102,False
8,1,BLEACH,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.721154,0.000000,0.721154,als,ALS,102,False
14,1,CHRISTMAS SEASONAL,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.523417,0.000000,0.523417,als,ALS,102,False



Household 2


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source,source_detail,snapshot_week,was_recently_purchased
86,2,DELI MEATS,APPLES | BAG SNACKS | BAKED BREAD/BUNS/ROLLS,0.00000,1.0,1.0,mba,MBA,102,False
79,2,CITRUS,APPLES | BAG SNACKS | BAKED BREAD/BUNS/ROLLS,0.29181,1.0,1.0,both,BOTH,102,False
103,2,LUNCHMEAT,APPLES | BAG SNACKS | BAKED BREAD/BUNS/ROLLS,0.00000,1.0,1.0,mba,MBA,102,False
120,2,TOMATOES,APPLES | BAG SNACKS | BAKED BREAD/BUNS/ROLLS,0.00000,1.0,1.0,mba,MBA,102,False
114,2,SANDWICHES,APPLES | BAG SNACKS | BAKED BREAD/BUNS/ROLLS,0.00000,1.0,1.0,mba,MBA,102,False



Household 3


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source,source_detail,snapshot_week,was_recently_purchased
138,3,CHICKEN,BAG SNACKS | BEEF | BREAKFAST SAUSAGE/SANDWICHES,0.0,1.0,1.0,mba,MBA,102,False
164,3,LAUNDRY DETERGENTS,BAG SNACKS | BEEF | BREAKFAST SAUSAGE/SANDWICHES,1.0,0.0,1.0,als,ALS,102,False
173,3,POTATOES,BAG SNACKS | BEEF | BREAKFAST SAUSAGE/SANDWICHES,0.0,1.0,1.0,mba,MBA,102,False
183,3,TOMATOES,BAG SNACKS | BEEF | BREAKFAST SAUSAGE/SANDWICHES,0.0,1.0,1.0,mba,MBA,102,False
179,3,SANDWICHES,BAG SNACKS | BEEF | BREAKFAST SAUSAGE/SANDWICHES,0.0,1.0,1.0,mba,MBA,102,False


In [8]:
if not candidate_set.empty:
    avg_candidates = candidate_set.groupby("household_key")["COMMODITY_DESC"].nunique().mean()
    print("1) Avg candidates per user:", round(float(avg_candidates), 2), "(target: 100-200)")

    source_pct = candidate_set["source_detail"].value_counts(normalize=True).mul(100).round(2)
    print("\n2) Source composition (%):")
    for key in ["ALS", "MBA", "BOTH"]:
        print(f" - {key}: {source_pct.get(key, 0.0)}%")

    removed_count = len(filtered_items_log)
    pre_filter_count = len(candidate_set) + len(filtered_items_log)
    removed_pct = (removed_count / pre_filter_count * 100.0) if pre_filter_count > 0 else 0.0
    print("\n3) Removed due to recency filter:", round(removed_pct, 2), "%")

    leak_check = candidate_set.merge(
        recent_pairs[["household_key", "COMMODITY_DESC"]],
        on=["household_key", "COMMODITY_DESC"],
        how="inner",
    )
    print("\n4) Leakage check (must be 0):", len(leak_check))

    print("\nCandidate count per user quantiles:")
    coverage = candidate_set.groupby("household_key")["COMMODITY_DESC"].nunique()
    print(coverage.quantile([0.0, 0.25, 0.5, 0.75, 1.0]).round(2))

    for column in ["relevance_als", "relevance_mba", "Relevance"]:
        low_ok = bool((candidate_set[column] >= 0.0).all())
        high_ok = bool((candidate_set[column] <= 1.0).all())
        print(f"{column}: within [0,1] ->", low_ok and high_ok)

    print("\nSample top-5 recommendations for 3 users:")
    for hh in candidate_set["household_key"].drop_duplicates().head(3):
        print(f"\nHousehold {hh}")
        display(candidate_set[candidate_set["household_key"] == hh].sort_values("Relevance", ascending=False).head(5))

1) Avg candidates per user: 47.02 (target: 100-200)

2) Source composition (%):
 - ALS: 68.06%
 - MBA: 17.87%
 - BOTH: 12.63%

3) Removed due to recency filter: 21.98 %

4) Leakage check (must be 0): 0

Candidate count per user quantiles:
0.00    10.0
0.25    38.0
0.50    48.0
0.75    55.0
1.00    82.0
Name: COMMODITY_DESC, dtype: float64
relevance_als: within [0,1] -> True
relevance_mba: within [0,1] -> True
Relevance: within [0,1] -> True

Sample top-5 recommendations for 3 users:

Household 1


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source,source_detail,snapshot_week,was_recently_purchased
59,1,VEGETABLES - ALL OTHERS,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.000000,1.000000,1.000000,mba,MBA,102,False
47,1,POPCORN,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.947021,0.000000,0.947021,als,ALS,102,False
7,1,BEEF,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.000000,0.796527,0.796527,mba,MBA,102,False
8,1,BLEACH,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.721154,0.000000,0.721154,als,ALS,102,False
14,1,CHRISTMAS SEASONAL,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.523417,0.000000,0.523417,als,ALS,102,False



Household 2


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source,source_detail,snapshot_week,was_recently_purchased
86,2,DELI MEATS,APPLES | BAG SNACKS | BAKED BREAD/BUNS/ROLLS,0.00000,1.0,1.0,mba,MBA,102,False
79,2,CITRUS,APPLES | BAG SNACKS | BAKED BREAD/BUNS/ROLLS,0.29181,1.0,1.0,both,BOTH,102,False
103,2,LUNCHMEAT,APPLES | BAG SNACKS | BAKED BREAD/BUNS/ROLLS,0.00000,1.0,1.0,mba,MBA,102,False
120,2,TOMATOES,APPLES | BAG SNACKS | BAKED BREAD/BUNS/ROLLS,0.00000,1.0,1.0,mba,MBA,102,False
114,2,SANDWICHES,APPLES | BAG SNACKS | BAKED BREAD/BUNS/ROLLS,0.00000,1.0,1.0,mba,MBA,102,False



Household 3


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source,source_detail,snapshot_week,was_recently_purchased
138,3,CHICKEN,BAG SNACKS | BEEF | BREAKFAST SAUSAGE/SANDWICHES,0.0,1.0,1.0,mba,MBA,102,False
164,3,LAUNDRY DETERGENTS,BAG SNACKS | BEEF | BREAKFAST SAUSAGE/SANDWICHES,1.0,0.0,1.0,als,ALS,102,False
173,3,POTATOES,BAG SNACKS | BEEF | BREAKFAST SAUSAGE/SANDWICHES,0.0,1.0,1.0,mba,MBA,102,False
183,3,TOMATOES,BAG SNACKS | BEEF | BREAKFAST SAUSAGE/SANDWICHES,0.0,1.0,1.0,mba,MBA,102,False
179,3,SANDWICHES,BAG SNACKS | BEEF | BREAKFAST SAUSAGE/SANDWICHES,0.0,1.0,1.0,mba,MBA,102,False


## Handoff to Module 3 (Utility Ranking)

This notebook now exports a reproducible and explainable Module 2 artifact set:

- `association_rules.csv`: normalized MBA signal (`relevance_mba`).
- `als_scores.parquet`: top-100 ALS dot-product scores per user, normalized to `[0,1]`.
- `user_factors.npz`: ALS user latent factors + `user_id_to_index` mapping metadata.
- `item_factors.npz`: ALS item latent factors + `item_id_to_index` mapping metadata.
- `candidate_set_module2.csv`: recall-union candidates with source traceability fields.
- `filtered_items_log.csv`: explicit audit log of recency-based removals.
- `user_item_matrix.npz`: sparse confidence matrix cache.

Candidate set columns include:
- `relevance_als`
- `relevance_mba`
- `Relevance = max(relevance_als, relevance_mba)`
- `source_detail` in `{ALS, MBA, BOTH}`
- `snapshot_week`
- `was_recently_purchased`

Module 3 can directly consume `candidate_set_module2.csv` and combine with margin, uplift, and context features.

$$
U(i,u)=w_1\cdot Relevance(i,u)+w_2\cdot Uplift(i,u)+w_3\cdot Margin(i)+w_4\cdot Context(i,u)
$$

with baseline weights:
- $w_1=0.40$
- $w_2=0.25$
- $w_3=0.20$
- $w_4=0.15$.